In [ ]:
# Loads cleaned CSV → runs through multilingual-e5-base → saves embeddings_en.npy + embeddings_fr.npy

In [1]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
import os
import time

In [2]:
data_path = "D:/Harshita Ajmani/Code_harshu/NLP/data/cleaned_data.csv"
save_path = "D:/Harshita Ajmani/Code_harshu/NLP/data/"
model = "intfloat/multilingual-e5-base"
batch_size = 64

#instead of encoding one record at a time, it processes 64 records simultaneously, which is much faster on a GPU

In [3]:
#Loading data

df = pd.read_csv(data_path)
print(f"Data Loaded {len(df):,} records")


Data Loaded 46,468 records


In [ ]:
#Building Searchable Text (Passages)
# each passage is a combination of the title, description, and keywords for that record, concatenated into a single string. 

def build_passage(row, lang):
    title    = row[f'title_{lang}']    or ""
    desc     = row[f'desc_{lang}']     or ""
    keywords = row[f'keywords_{lang}'] or ""
    text = f"{title}. {desc} {keywords}".strip()
    return f"passage: {text}"

df['passage_en'] = df.apply(lambda row: build_passage(row, 'en'), axis=1)
df['passage_fr'] = df.apply(lambda row: build_passage(row, 'fr'), axis=1)

print(f"Sample EN passage:\n   {df['passage_en'].iloc[0][:150]}...")
print(f"Sample FR passage:\n   {df['passage_fr'].iloc[0][:150]}...")

Sample EN passage:
   passage: Principal Mineral Areas, Producing Mines, and Oil and Gas Fields (900A). This dataset is produced and published annually by Natural Resources Canada. It contains a variety of statistics on Canada’s mineral production, and provides the geographic locations of significant metallic, nonmetallic and coal mines, oil sands mines, selected metallurgical works, helium facilities, and oil and gas fields for the provinces and territories of Canada. Related product: - mineralization, mineral occurrences, mines, hydrocarbons, fossil fuels, industrial minerals, metallic minerals, economic geology, mineral deposits, exploration and deposit appraisal, nonmetallic minerals, oil, gas, hydrocarbons, refineries, smelters, mineral processing, ferroalloy, automobile shredders, helium, Coal, Earth sciences, Oil sands, Gas industry, Mining industry, Steel, Metals, Minerals, Recycling...
Sample FR passage:
   passage: Principales régions minières, principales mines productrices,

In [5]:
# Load the model

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device} ({torch.cuda.get_device_name(0)})")    
model = SentenceTransformer(model, device=device)
print("Model loaded successfully.")


Using device: cuda (NVIDIA GeForce RTX 5070 Laptop GPU)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully.


In [6]:
#Generating English Embeddings

start = time.time()

embeddings_en = model.encode(
    df['passage_en'].tolist(),
    batch_size = batch_size,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(f"English embeddings done in {time.time() - start:.1f}s")
print(f"Shape: {embeddings_en.shape}")

# normalize_embeddings=True
# Scales each vector to length 1 — required for cosine similarity to work correctly

Batches:   0%|          | 0/727 [00:00<?, ?it/s]

English embeddings done in 246.7s
Shape: (46468, 768)


In [8]:
# Generating French Embeddings

start = time.time()

embeddings_fr = model.encode(
    df['passage_fr'].tolist(),
    batch_size = batch_size,
    show_progress_bar=True,
    normalize_embeddings=True
)
print(f"French embeddings done in {time.time() - start:.1f}s")
print(f"Shape: {embeddings_fr.shape}")


Batches:   0%|          | 0/727 [00:00<?, ?it/s]

French embeddings done in 241.9s
Shape: (46468, 768)


In [9]:
# Saving the embeddings
# index_data.csv contains the original metadata for each record, which will be used later to display search results (titles, descriptions, etc.) 
# when a user clicks on a search result in the UI.

np.save(os.path.join(save_path, "embeddings_en.npy"), embeddings_en)
np.save(os.path.join(save_path, "embeddings_fr.npy"), embeddings_fr)

df[['id', 'title_en', 'title_fr', 'desc_en', 'desc_fr', 'keywords_en', 'keywords_fr', 'subject', 'org']].to_csv(os.path.join(save_path, "index_data.csv"), index=False)

print(f"Saved embeddings_en.npy  → {embeddings_en.nbytes / 1e6:.1f} MB")
print(f"Saved embeddings_fr.npy  → {embeddings_fr.nbytes / 1e6:.1f} MB")
print(f"Saved index_data.csv")

Saved embeddings_en.npy  → 142.7 MB
Saved embeddings_fr.npy  → 142.7 MB
Saved index_data.csv
